# 🎓 E.S.I.S (Electronic Smoking Identification System)
## Modul AI Face Recognition: MTCNN + FaceNet (L2 Normalized & CCTV Optimized)
**Metodologi:** CRISP-DM (*Cross-Industry Standard Process for Data Mining*)  
**Tujuan:** Mengidentifikasi identitas mahasiswa/pelaku yang tertangkap kamera CCTV saat aktivitas merokok terdeteksi.

---
### 📌 Kunci Keberhasilan Deteksi Wajah pada CCTV:
1. **MTCNN Confidence Threshold (0.80):**  
   - Wajah pada kamera CCTV seringkali berjarak jauh, sedikit buram, atau memiliki sudut miring (*overhead angle*). Ambang batas diatur ke **0.80** (bukan 0.90/0.95) agar wajah pada frame CCTV tidak diabaikan/dilewati.
2. **Proporsional Margin Padding (10%):**  
   - Memberi ruang 10% di sekeliling kotak deteksi agar dahi, telinga, dan dagu tidak terpotong ketat.
3. **Standarisasi L2-Normalization pada Vektor Embedding:**  
   - Setiap vektor embedding FaceNet 512-D **wajib di-normalisasi** (`emb = emb / np.linalg.norm(emb)`).
   - Tanpa normalisasi L2, panjang vektor FaceNet berkisar 8.0 - 12.0 sehingga jarak Euclidean antar-wajah membengkak dan selalu dianggap *Unknown*.
   - Dengan normalisasi L2, ambang batas jarak **Threshold = 0.75** bekerja secara akurat dan konsisten.
4. **Pencocokan 1-NN (Nearest Neighbor Euclidean Distance):**  
   - Membandingkan wajah target terhadap seluruh template foto wajah mahasiswa di database untuk akurasi pengenalan tertinggi.


In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT & DEPENDENCIES
# ==============================================================================
!pip install -q mtcnn keras-facenet opencv-python scikit-learn seaborn matplotlib tqdm

import os
import sys
import time
import shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image as Img
from numpy import asarray, expand_dims
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from mtcnn import MTCNN
from keras_facenet import FaceNet

print("✅ Seluruh dependency (MTCNN, FaceNet, OpenCV, Scikit-Learn) berhasil dimuat!")


---
## 📊 2. DATA UNDERSTANDING
Memuat dataset foto wajah mahasiswa volunteer dari Google Drive (`known_faces`) dan memvisualisasikan persebaran data per kelas.


In [ ]:
# ==============================================================================
# 2. MOUNT GOOGLE DRIVE & EKSPLORASI DATASET
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ SESUAIKAN DENGAN PATH DATASET WAJAH DI GOOGLE DRIVE ANDA
# Contoh: '/content/drive/MyDrive/DATASET/WAJAH/' atau '/content/drive/MyDrive/Skripsi/AI/known_faces'
dataset_raw_path = '/content/drive/MyDrive/DATASET/WAJAH/'

if not os.path.exists(dataset_raw_path):
    print(f"⚠️ Folder '{dataset_raw_path}' tidak ditemukan!")
    print("Silakan periksa path folder dataset Anda di Google Drive.")
else:
    class_names = []
    image_counts = []

    for folder in sorted(os.listdir(dataset_raw_path)):
        folder_path = os.path.join(dataset_raw_path, folder)
        if os.path.isdir(folder_path) and not folder.startswith('.'):
            valid_images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if len(valid_images) > 0:
                class_names.append(folder)
                image_counts.append(len(valid_images))

    print(f"Total Kelas Terdaftar : {len(class_names)} Mahasiswa/Volunteer")
    print(f"Total Keseluruhan Foto: {sum(image_counts)} gambar")

    # Visualisasi Distribusi Dataset per Kelas
    plt.figure(figsize=(15, 6))
    ax = sns.barplot(x=class_names, y=image_counts, hue=class_names, palette='magma', legend=False)
    plt.xlabel('Nama Mahasiswa / Kelas', fontsize=12, fontweight='bold')
    plt.ylabel('Jumlah Foto', fontsize=12, fontweight='bold')
    plt.title('Distribusi Jumlah Foto per Kelas (Dataset Wajah)', fontsize=15, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, max(image_counts) + 5)

    for container in ax.containers:
        ax.bar_label(container, padding=3, fmt='%d')

    plt.tight_layout()
    plt.show()


---
## 🧹 3. DATA PREPARATION (Deteksi & Pemotongan Wajah dengan MTCNN)
Pada tahap ini:
- Detektor **MTCNN** mendeteksi wajah dengan `confidence >= 0.80` (ramah terhadap sudut pandang CCTV).
- Diberikan *margin padding* 10% di setiap sisi bounding box agar dahi, telinga, dan dagu tidak terpotong.
- Wajah hasil potong diubah ukurannya ke **160x160 piksel** (standar input FaceNet).


In [ ]:
# ==============================================================================
# 3. KELAS DETEKTOR WAJAH MTCNN (OPTIMAL UNTUK CCTV)
# ==============================================================================
class MTCNNFaceDetector:
    """Wrapper untuk MTCNN Face Detector yang dioptimalkan untuk CCTV"""
    def __init__(self, min_confidence=0.80):
        self.min_confidence = min_confidence
        print("⏳ Menginisialisasi model MTCNN...")
        self.detector = MTCNN()
        print("✅ Model MTCNN siap!")

    def detect_face(self, img_bgr, margin_percent=0.10):
        """
        Mendeteksi wajah pada gambar BGR.
        Mengembalikan: (face_160_rgb, bbox) atau (None, None) jika tidak terdeteksi.
        """
        h, w, _ = img_bgr.shape
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        results = self.detector.detect_faces(img_rgb)
        if not results:
            return None, None

        # Ambil wajah dengan confidence tertinggi
        best_face = max(results, key=lambda f: f['confidence'])
        if best_face['confidence'] < self.min_confidence:
            return None, None

        x, y, fw, fh = best_face['box']
        x, y = abs(x), abs(y)

        # Padding 10%
        pad_x = int(fw * margin_percent)
        pad_y = int(fh * margin_percent)

        x1 = max(0, x - pad_x)
        y1 = max(0, y - pad_y)
        x2 = min(w, x + fw + pad_x)
        y2 = min(h, y + fh + pad_y)

        face_crop = img_rgb[y1:y2, x1:x2]
        if face_crop.size == 0:
            return None, None

        face_pil = Img.fromarray(face_crop).resize((160, 160))
        return np.asarray(face_pil), (x1, y1, x2 - x1, y2 - y1)

# Inisialisasi Detektor MTCNN
mtcnn_detector = MTCNNFaceDetector(min_confidence=0.80)


In [ ]:
# ==============================================================================
# 4. PREPROCESSING SELURUH DATASET KE FOLDER BERSIH (160x160)
# ==============================================================================
output_preprocessed_dir = '/content/DATA_PREPROCESSED_MTCNN'

if os.path.exists(output_preprocessed_dir):
    shutil.rmtree(output_preprocessed_dir)
os.makedirs(output_preprocessed_dir)

sample_visualizations = []
success_count = 0
fail_count = 0

print(f"🚀 Memulai Preprocessing dataset dengan MTCNN dari '{dataset_raw_path}'...")

for class_name in tqdm(class_names, desc="Memproses Mahasiswa"):
    class_src_dir = os.path.join(dataset_raw_path, class_name)
    class_dst_dir = os.path.join(output_preprocessed_dir, class_name)
    os.makedirs(class_dst_dir, exist_ok=True)

    has_sample = False

    for fname in os.listdir(class_src_dir):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        file_path = os.path.join(class_src_dir, fname)
        img = cv2.imread(file_path)
        if img is None:
            continue

        face_160, bbox = mtcnn_detector.detect_face(img, margin_percent=0.10)

        if face_160 is not None:
            save_path = os.path.join(class_dst_dir, fname)
            Img.fromarray(face_160).save(save_path)
            success_count += 1

            if not has_sample:
                sample_visualizations.append((face_160, class_name))
                has_sample = True
        else:
            fail_count += 1

print(f"\n✅ Preprocessing MTCNN Selesai!")
print(f"   - Wajah berhasil dipotong (160x160): {success_count} foto")
print(f"   - Foto gagal/tidak terdeteksi       : {fail_count} foto")

# Visualisasi Sampel Hasil Preprocessing
if sample_visualizations:
    cols = 5
    rows = (len(sample_visualizations) + cols - 1) // cols
    plt.figure(figsize=(15, 3 * rows))
    for i, (sample_img, label) in enumerate(sample_visualizations):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(sample_img)
        plt.title(label[:15], fontsize=9)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


---
## 🧠 4. MODELLING (Ekstraksi Fitur FaceNet dengan L2 Normalization)
Setiap wajah berukuran 160x160 diubah menjadi vektor representasi 512 dimensi menggunakan **FaceNet**.
Setiap vektor kemudian di-**L2 Normalized**:
$$\hat{v} = \frac{v}{\|v\|_2}$$
Hal ini memastikan seluruh vektor berada pada *unit hypersphere* sehingga perhitungan **Euclidean Distance** dengan ambang batas **Threshold = 0.75** berjalan akurat.


In [ ]:
# ==============================================================================
# 5. FEATURE EXTRACTION MENGGUNAKAN FACENET DENGAN L2 NORMALIZATION
# ==============================================================================
print("⏳ Memuat model FaceNet (Inception-ResNet-v1)... ")
facenet = FaceNet()
print("✅ FaceNet berhasil dimuat!")

X_embeddings = []
Y_labels = []

for class_name in sorted(os.listdir(output_preprocessed_dir)):
    class_dir = os.path.join(output_preprocessed_dir, class_name)
    if not os.path.isdir(class_dir):
        continue

    files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(files) == 0:
        continue

    for f in files:
        img_path = os.path.join(class_dir, f)
        try:
            face_img = Img.open(img_path).convert('RGB')
            face_arr = asarray(face_img)

            face_input = expand_dims(face_arr, axis=0)
            emb = facenet.embeddings(face_input)[0]

            # ⚠️ KUNCI UTAMA: L2 NORMALIZATION
            emb = emb / np.linalg.norm(emb)

            X_embeddings.append(emb)
            Y_labels.append(class_name)
        except Exception as e:
            print(f"Error {f}: {e}")

X_embeddings = np.array(X_embeddings)
Y_labels = np.array(Y_labels)

print(f"\n✅ Total Embedding Terbentuk : {X_embeddings.shape[0]} vektor wajah")
print(f"   Dimensi per Vektor Wajah : {X_embeddings.shape[1]} fitur numerik")

# Simpan Basis Data Embedding Wajah ke format .npz
npz_output_path = "data_wajah_facenet_mtcnn.npz"
np.savez_compressed(npz_output_path, x=X_embeddings, y=Y_labels)
print(f"💾 Basis data embedding disimpan ke: '{npz_output_path}'")


---
## 📈 5. EVALUATION
Pengujian performa pencocokan wajah menggunakan **Nearest Neighbor Euclidean Distance** dengan ambang batas **THRESHOLD = 0.75**:
- Jika $\text{Jarak} < 0.75$: Wajah dikenali sesuai label dengan persentase kemiripan $\ge 60\%$.
- Jika $\text{Jarak} \ge 0.75$: Diklasifikasikan sebagai *Unknown* (Bukan mahasiswa terdaftar).


In [ ]:
# ==============================================================================
# 6. EVALUASI AKURASI DENGAN JARAK EUCLIDEAN TERL-2-NORMALISASI
# ==============================================================================
THRESHOLD = 0.75

y_true = []
y_pred = []

# Uji coba leave-one-out / nearest neighbor pada seluruh dataset
print("⏳ Menghitung matriks evaluasi pengenalan...")
for i in range(len(X_embeddings)):
    target_emb = X_embeddings[i]
    true_label = Y_labels[i]

    # Hitung jarak ke seluruh database kecuali foto dirinya sendiri
    distances = []
    for j in range(len(X_embeddings)):
        if i == j:
            distances.append(999.0) # Abaikan vektor identik dirinya sendiri
        else:
            d = np.linalg.norm(target_emb - X_embeddings[j])
            distances.append(d)

    min_dist = min(distances)
    best_idx = distances.index(min_dist)

    if min_dist < THRESHOLD:
        pred_label = Y_labels[best_idx]
    else:
        pred_label = "Unknown"

    y_true.append(true_label)
    y_pred.append(pred_label)

acc = accuracy_score(y_true, y_pred)
print("=" * 60)
print(f"🎯 AKURASI PENGENALAN WAJAH (MTCNN + FACENET L2): {acc * 100:.2f}%")
print("=" * 60)

print("\n📋 Laporan Klasifikasi:")
print(classification_report(y_true, y_pred, zero_division=0))

# Confusion Matrix
unique_labels = sorted(list(set(y_true + y_pred)))
cm = confusion_matrix(y_true, y_pred, labels=unique_labels)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=unique_labels,
            yticklabels=unique_labels)
plt.title(f'Confusion Matrix Pengenalan Wajah (Threshold {THRESHOLD})\nAkurasi: {acc*100:.2f}%', fontsize=14, fontweight='bold')
plt.xlabel('Prediksi AI', fontsize=12)
plt.ylabel('Label Sebenarnya', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


---
## 🔍 6. FUNGSI PENGENALAN WAJAH (SAMA PERSIS DENGAN PIPELINE CCTV)
Fungsi `kenali_wajah_cctv` di bawah ini menggunakan logika yang sama persis dengan yang berhasil berjalan di kamera CCTV Anda kemarin:
1. Membaca gambar/frame BGR $\to$ konversi ke RGB.
2. Deteksi wajah MTCNN dengan threshold $\ge 0.80$.
3. Crop wajah dengan padding 10% $\to$ Resize ke 160x160.
4. Ekstraksi embedding FaceNet $\to$ **L2-Normalize**.
5. Menghitung jarak Euclidean terhadap database `data_wajah_facenet_mtcnn.npz`.
6. Jika jarak $< 0.75$, menampilkan kotak hijau dan nama mahasiswa + persentase kemiripan!


In [ ]:
# ==============================================================================
# 7. FUNGSI INFERENSI LENGKAP (CCTV COMPATIBLE)
# ==============================================================================
def kenali_wajah_cctv(image_path_or_frame, threshold=0.75):
    """
    Fungsi pengenalan wajah untuk frame CCTV maupun foto lokal.
    """
    if isinstance(image_path_or_frame, str):
        img_bgr = cv2.imread(image_path_or_frame)
        if img_bgr is None:
            print(f"❌ Gagal membaca gambar: {image_path_or_frame}")
            return
    else:
        img_bgr = image_path_or_frame.copy()

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    t_start = time.time()
    results = mtcnn_detector.detector.detect_faces(img_rgb)
    t_detect = (time.time() - t_start) * 1000

    if not results:
        print("❌ Tidak ada wajah yang terdeteksi.")
        return

    plt.figure(figsize=(9, 7))
    plt.imshow(img_rgb)
    ax = plt.gca()

    for face_info in results:
        if face_info['confidence'] < 0.80:
            continue

        x, y, fw, fh = face_info['box']
        x, y = abs(x), abs(y)

        # Padding 10%
        m_x, m_y = int(fw * 0.1), int(fh * 0.1)
        x1, y1 = max(0, x - m_x), max(0, y - m_y)
        x2, y2 = min(w, x + fw + m_x), min(h, y + fh + m_y)

        crop = img_rgb[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        face_pil = Img.fromarray(crop).resize((160, 160))
        face_input = expand_dims(asarray(face_pil).astype(np.float32), axis=0)

        t_emb_start = time.time()
        emb_target = facenet.embeddings(face_input)[0]
        # L2-NORMALIZATION
        emb_target = emb_target / np.linalg.norm(emb_target)
        t_emb = (time.time() - t_emb_start) * 1000

        # Hitung Jarak Euclidean terhadap seluruh database
        dists = [np.linalg.norm(emb_target - db_emb) for db_emb in X_embeddings]
        min_dist = min(dists)
        best_idx = dists.index(min_dist)

        if min_dist < threshold:
            nama = Y_labels[best_idx]
            sim = max(0, (1 - (min_dist / threshold))) * 100
            label_text = f"{nama} ({sim:.1f}%)"
            box_color = 'green'
        else:
            label_text = f"Unknown ({min_dist:.2f})"
            box_color = 'red'

        rect = plt.Rectangle((x, y), fw, fh, fill=False, color=box_color, linewidth=2.5)
        ax.add_patch(rect)
        plt.text(x, max(20, y - 8), label_text, color='white', fontsize=11, fontweight='bold',
                 bbox=dict(facecolor=box_color, alpha=0.75, edgecolor='none', boxstyle='round,pad=0.4'))

    plt.axis('off')
    plt.title(f"Deteksi MTCNN: {t_detect:.1f}ms | FaceNet Embed: {t_emb:.1f}ms", fontsize=12, fontweight='bold')
    plt.show()

# Uji pada 1 sampel foto dari dataset hasil cropping
sample_img = None
for root, _, f_list in os.walk(output_preprocessed_dir):
    for fn in f_list:
        if fn.lower().endswith(('.jpg', '.png')):
            sample_img = os.path.join(root, fn)
            break
    if sample_img: break

if sample_img:
    print(f"Menguji sampel foto: {sample_img}")
    kenali_wajah_cctv(sample_img)


---
## 📷 7. PENGUJIAN LANGSUNG DARI WEBCAM (GOOGLE COLAB)
Uji coba pengenalan wajah langsung dari kamera laptop Anda menggunakan tombol *Capture*.


In [ ]:
# ==============================================================================
# 8. CAPTURE WEBCAM DI GOOGLE COLAB
# ==============================================================================
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def capture_colab_photo(filename='webcam_test_mtcnn.jpg', quality=0.85):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Ambil Foto Wajah (MTCNN)';
      capture.style.padding = '10px 20px';
      capture.style.fontSize = '16px';
      capture.style.backgroundColor = '#2E7D32';
      capture.style.color = 'white';
      capture.style.border = 'none';
      capture.style.borderRadius = '8px';
      capture.style.cursor = 'pointer';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.marginTop = '10px';
      video.style.borderRadius = '8px';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

try:
    print("Silakan klik tombol '📸 Ambil Foto Wajah (MTCNN)' yang muncul di bawah:")
    test_photo_file = capture_colab_photo()
    print("✅ Foto webcam berhasil diambil! Menganalisis wajah...")
    kenali_wajah_cctv(test_photo_file)
except Exception as e:
    print("Info:", e)


---
## 💾 8. UNDUH ARTIFAK MODEL UNTUK DEPLOYMENT KE AI SERVER
Unduh berkas database vektor embedding wajah untuk dipakai di AI Server:
- `data_wajah_facenet_mtcnn.npz`


In [ ]:
# ==============================================================================
# 9. DOWNLOAD BERKAS DATABASE WAJAH (.NPZ)
# ==============================================================================
from google.colab import files

npz_file = "data_wajah_facenet_mtcnn.npz"
if os.path.exists(npz_file):
    print(f"Mengunduh berkas database: {npz_file}...")
    try:
        files.download(npz_file)
    except Exception as e:
        print(f"Gagal auto-download {npz_file}: {e}")
else:
    print(f"⚠️ Berkas {npz_file} belum tersedia.")
